##Cell 1 — Install + check GPU

In [1]:
# Cell 1 — Install + check GPU + reproducibility setup

from google.colab import files
uploaded = files.upload()  # Upload raft_clean.jsonl (and optionally a pinned requirements file later)

# Install core packages (current run)
!pip -q install unsloth trl datasets accelerate
# Optional: avoids some audio dependency conflicts/warnings
!pip -q uninstall -y torchaudio

# -----------------------------
# Reproducibility: set seeds
# -----------------------------
import os, random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(SEED)
np.random.seed(SEED)

# Torch is imported after install
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make some operations more deterministic (best effort)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("SEED:", SEED)

# -----------------------------
# Reproducibility: record versions
# -----------------------------
!python -V
!pip show torch transformers trl unsloth datasets accelerate | tee colab_versions.txt
!pip freeze | sort > colab_pip_freeze.txt

print("✅ Saved version logs: colab_versions.txt, colab_pip_freeze.txt")


Saving raft_clean.jsonl to raft_clean.jsonl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.4 MB/s eta 0:00:

##Cell 2 — Load + format dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="raft_clean.jsonl", split="train")
print("rows:", len(dataset))
print("cols:", dataset.column_names)
print("sample:", dataset[0])

# Use the SAME style as app.py / scripts/5_rag_cli.py
# Training prompt text now matches runtime prompt shape much better.
def format_example(ex):
    return {
        "text": (
            "You are a medical document QA assistant.\n"
            "RULES:\n"
            "- Use ONLY the SOURCES below.\n"
            "- If the answer is not clearly supported by the sources, say: "
            "\"I don't know based on the provided documents.\"\n"
            "- In your answer, cite sources like [S1], [S2] next to the claims they support.\n"
            "- Keep the answer concise and factual.\n\n"
            "QUESTION:\n"
            + ex["instruction"].strip() + "\n\n"
            "SOURCES:\n"
            + ex["input"].strip() + "\n\n"
            "ANSWER:\n"
            + ex["output"].strip()
        )
    }

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(dataset[0]["text"][:1000])


Generating train split: 0 examples [00:00, ? examples/s]

rows: 49
cols: ['instruction', 'input', 'output']
sample: {'instruction': 'What is the recommendation for individuals who smoke in relation to smoking cessation?', 'input': 'Context (oracle):\n## Smoking cessation\n\n## Lifestyle interventions for the management of type 2 diabetes | Smoking cessation\n\n## Table of recommendations\n\n| Recommendation                                                                      | Grade                |   References | Recommended as of:   |\n|-------------------------------------------------------------------------------------|----------------------|--------------|----------------------|\n| All people who smoke should be offered brief advice and medications to quit smoking | Recommended (Strong) |            1 | 14/11/2024           |\n\n## Clinical context\n\nDistractors:\nSetting clinical target recommendations should consider both the individual and population-level impact. The global prevalence of poor glycaemic control is high, ranging from 

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

You are a medical document QA assistant.
RULES:
- Use ONLY the SOURCES below.
- If the answer is not clearly supported by the sources, say: "I don't know based on the provided documents."
- In your answer, cite sources like [S1], [S2] next to the claims they support.
- Keep the answer concise and factual.

QUESTION:
What is the recommendation for individuals who smoke in relation to smoking cessation?

SOURCES:
Context (oracle):
## Smoking cessation

## Lifestyle interventions for the management of type 2 diabetes | Smoking cessation

## Table of recommendations

| Recommendation                                                                      | Grade                |   References | Recommended as of:   |
|-------------------------------------------------------------------------------------|----------------------|--------------|----------------------|
| All people who smoke should be offered brief advice and medications to quit smoking | Recommended (Strong) |            1 | 14/11/

##Cell 3 — Load model + LoRA

In [3]:
from unsloth import FastLanguageModel

print("Using SEED =", SEED)

max_seq_length = 2048
dtype = None
load_in_4bit = True  # saves VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# If your installed unsloth version supports random_state, keep it.
# If it errors, remove random_state=SEED and rerun this cell.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    random_state=SEED,   # <-- optional but helpful if supported
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Using SEED = 42
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2026.2.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


##Cell 4 — Train

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling, set_seed

# Set Transformers seed too (important)
set_seed(SEED)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        warmup_steps=10,
        max_steps=300,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        save_steps=100,

        # Reproducibility / cleaner runs
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
        report_to="none",   # avoids wandb prompt
    ),
)

train_result = trainer.train()
print("✅ Training finished")
print(train_result)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/49 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 75 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,2.361500
20,2.029000
30,1.785000
40,1.662700
50,1.542300
60,1.368300
70,1.256100
80,1.053900
90,0.915100
100,0.715100


✅ Training finished
TrainOutput(global_step=300, training_loss=0.5567997014522552, metrics={'train_runtime': 710.9133, 'train_samples_per_second': 6.752, 'train_steps_per_second': 0.422, 'total_flos': 7774282476583680.0, 'train_loss': 0.5567997014522552, 'epoch': 75.0})


##Cell 5 — Quick inference test (sanity check)

In [5]:
# Cell 5 — Quick inference test (sanity check) using app/CLI-style prompt

FastLanguageModel.for_inference(model)

prompt = """You are a medical document QA assistant.
RULES:
- Use ONLY the SOURCES below.
- If the answer is not clearly supported by the sources, say: "I don't know based on the provided documents."
- In your answer, cite sources like [S1], [S2] next to the claims they support.
- Keep the answer concise and factual.

QUESTION:
What treatment is recommended in this example?

SOURCES:
[S1] pdf=example_guideline chunk_id=example_chunk_1
Hypertension guideline: start ACE inhibitor at X dose...

[S2] pdf=example_guideline chunk_id=example_chunk_2
Distractor text not relevant to the question.

ANSWER:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.2,
)
print(tokenizer.decode(out[0], skip_special_tokens=True))


You are a medical document QA assistant.
RULES:
- Use ONLY the SOURCES below.
- If the answer is not clearly supported by the sources, say: "I don't know based on the provided documents."
- In your answer, cite sources like [S1], [S2] next to the claims they support.
- Keep the answer concise and factual.

QUESTION:
What treatment is recommended in this example?

SOURCES:
[S1] pdf=example_guideline chunk_id=example_chunk_1
Hypertension guideline: start ACE inhibitor at X dose...

[S2] pdf=example_guideline chunk_id=example_chunk_2
Distractor text not relevant to the question.

ANSWER:
The suggested treatment for pre-existing (non-diabetes) hypertension is an ACE inhibitor because of the RACGP recommendation when screening patients for hypertension, consider screening when a diagnosis of diabetes is confirmed.

Final decision-making is a collaborative process involving all members of the care team.

Distractors text not relevant to the question.

<footer>Subject line: Guide summary - Al

##Cell 6 — Save adapter (LoRA)

In [6]:
model.save_pretrained("raft_lora_adapter")
tokenizer.save_pretrained("raft_lora_adapter")

# Add reproducibility metadata file
import json
from pathlib import Path
import platform

meta = {
    "base_model": "unsloth/Qwen2.5-0.5B-Instruct",
    "seed": SEED,
    "max_seq_length": max_seq_length,
    "load_in_4bit": load_in_4bit,
    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0.0,
        "target_modules": [
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj",
        ],
    },
    "training": {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 16,
        "warmup_steps": 10,
        "max_steps": 300,
        "learning_rate": 2e-4,
        "fp16": True,
    },
    "dataset": {
        "rows": len(dataset),
        "text_field": "text",
        "source_file_uploaded": "raft_clean.jsonl",
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
}

Path("raft_lora_adapter/train_run_meta.json").write_text(
    json.dumps(meta, indent=2), encoding="utf-8"
)

print("✅ Saved metadata: raft_lora_adapter/train_run_meta.json")


✅ Saved metadata: raft_lora_adapter/train_run_meta.json


##Cell 7 — OPTIONAL: merge + convert to GGUF + quantize

In [8]:
# Merge LoRA into a full 16-bit model (optional, needed for GGUF conversion)
model.save_pretrained_merged("merged_model_16bit", tokenizer, save_method="merged_16bit")

!rm -rf /content/llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

# Pin llama.cpp to the exact working commit (reproducible)
LLAMA_CPP_COMMIT = "244641955f6146f7e8474afff7772d427593a534"
!cd /content/llama.cpp && git checkout {LLAMA_CPP_COMMIT}

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model_16bit \
  --outfile /content/qwen2.5-0.5b-raft-f16.gguf --outtype f16

!ls -lh /content/qwen2.5-0.5b-raft-f16.gguf

from google.colab import files
files.download("/content/qwen2.5-0.5b-raft-f16.gguf")

# Build quantizer + quantize to Q8_0
!rm -rf /content/llama.cpp/build
!cmake -S /content/llama.cpp -B /content/llama.cpp/build \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLAMA_BUILD_TESTS=OFF \
  -DLLAMA_BUILD_EXAMPLES=OFF

!cmake --build /content/llama.cpp/build --target llama-quantize -j 1

!/content/llama.cpp/build/bin/llama-quantize \
  /content/qwen2.5-0.5b-raft-f16.gguf \
  /content/qwen2.5-0.5b-raft-q8_0.gguf \
  q8_0

!ls -lh /content/qwen2.5-0.5b-raft-q8_0.gguf
files.download("/content/qwen2.5-0.5b-raft-q8_0.gguf")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 6754.11it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 11.87s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model_16bit`
Cloning into 'llama.cpp'...
remote: Enumerating objects: 81026, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 81026 (delta 143), reused 77 (delta 76), pack-reused 80821 (from 3)
Receiving objects: 100% (81026/81026), 306.60 MiB | 33.89 MiB/s, done.
Resolving deltas: 100% (58534/58534), done.
Note: switching to '244641955f6146f7e8474afff7772d427593a534'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detac

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 8 — Export verification + metadata summary

In [10]:
import os, json

build_info = {
    "base_model_family": "Qwen2.5-0.5B-Instruct (LoRA fine-tuned)",
    "llama_cpp_commit": "244641955f6146f7e8474afff7772d427593a534",
    "f16_gguf": "/content/qwen2.5-0.5b-raft-f16.gguf",
    "q8_0_gguf": "/content/qwen2.5-0.5b-raft-q8_0.gguf",
    "f16_exists": os.path.exists("/content/qwen2.5-0.5b-raft-f16.gguf"),
    "q8_0_exists": os.path.exists("/content/qwen2.5-0.5b-raft-q8_0.gguf"),
    "f16_size_mb": round(os.path.getsize("/content/qwen2.5-0.5b-raft-f16.gguf") / (1024**2), 2) if os.path.exists("/content/qwen2.5-0.5b-raft-f16.gguf") else None,
    "q8_0_size_mb": round(os.path.getsize("/content/qwen2.5-0.5b-raft-q8_0.gguf") / (1024**2), 2) if os.path.exists("/content/qwen2.5-0.5b-raft-q8_0.gguf") else None,
}

with open("/content/gguf_build_info.json", "w") as f:
    json.dump(build_info, f, indent=2)

print(json.dumps(build_info, indent=2))

{
  "base_model_family": "Qwen2.5-0.5B-Instruct (LoRA fine-tuned)",
  "llama_cpp_commit": "244641955f6146f7e8474afff7772d427593a534",
  "f16_gguf": "/content/qwen2.5-0.5b-raft-f16.gguf",
  "q8_0_gguf": "/content/qwen2.5-0.5b-raft-q8_0.gguf",
  "f16_exists": true,
  "q8_0_exists": true,
  "f16_size_mb": 948.1,
  "q8_0_size_mb": 506.47
}
